# Introduction to the IISG Base Groundwater Model

## Overview

This notebook introduces you to the **Illinois-Indiana SeaGrant (IISG) simplified groundwater model** — a regional MODFLOW 6 simulation of North East Illinois' Sand and Gravel aquifer system.

### What you will learn
1. How to **load** a MODFLOW 6 simulation using [FloPy](https://flopy.readthedocs.io/en/stable/)
2. How to **inspect** key model inputs (hydraulic conductivity, layer geometry, heads)
3. How to **visualize** the model in plan-view maps and vertical cross-sections
4. How to compute **Transmissivity** from model properties

### What is [MODFLOW 6](https://www.usgs.gov/software/modflow-6-usgs-modular-hydrologic-model)?
MODFLOW 6 (MF6) is the U.S. Geological Survey's current generation groundwater modeling code. It represents the subsurface as a three-dimensional grid of cells and solves for groundwater heads using the [finite-difference method](https://fc79.gw-project.org/english/appendices/) [see Appendix IX].

### What is [FloPy](https://flopy.readthedocs.io/en/stable/)?
FloPy is a Python package that acts as an interface to MODFLOW. It lets you load, modify, run, and post-process MODFLOW models entirely within [Python](https://www.python.org/).

### What does a Flopy/MF6 workflow look like?
![Alt Text](https://camo.githubusercontent.com/2198d99604c50858384e8238c6cc0605b8159199cba780a5ec1459404967032b/68747470733a2f2f692e696d6775722e636f6d2f6e32446a3651462e706e67)

Thank you **[Shelby](https://github.com/shelbyahrendt)**!!

## I. Imports

We begin by importing the Python libraries we will use throughout this notebook:

| Library | Purpose |
|---------|---------|
| `pathlib.Path` | Cross-platform file and directory path handling |
| `numpy` | Numerical array operations |
| `matplotlib.pyplot` | Plotting and visualization |
| `flopy` | MODFLOW 6 interface (load, modify, post-process models) |

In [ ]:
# Standard Library Imports
from pathlib import Path

# Scientific Stack
import numpy as np
import matplotlib.pyplot as plt

# Domain-Specific Packages (Flopy)
import flopy

## II. Housekeeping

Before loading any data, we establish a consistent set of file paths and constants.
Using `pathlib.Path` ensures these paths work correctly on Windows, macOS, and Linux.

The directory structure used in this project is:

```
home_dir/
├──bin/                  ← MODFLOW 6 executable[s]
├──config/
├──modflow_models/       ← MODFLOW 6 models will be written and read from here
│   └──iisg_local_model   ← base model (steady-state)
│       ├──external
│       ├──postproc
│       │   ├───pdfs
│       │   ├───rasters
│       │   └───shps
│       └──tables
├───python_notebooks/    ← this notebook lives here
└───shapefiles/
```

In [ ]:
# Use pathlib to get this notebook's location, and Python's "current working directory" (cwd)

notebook_dir = Path.cwd()

print(notebook_dir)

In [ ]:
# Define a home directory one level up from our cwd, so we can easily refer to our base model,
# create our updated model, and access other helpful files that "live outside" the working directory

home_dir = notebook_dir.parent

print(home_dir)

In [ ]:
# List the top-level contents of the home directory to confirm the expected structure
for item in home_dir.iterdir():
    print(item.name)

In [ ]:
# Other housekeeping constants and paths

sim_name = "mfsim" # Name of the simulation containing our base model

base_model_workspace = home_dir / "modflow_models" / "iisg_local_model" # Path to base model

exe_name = home_dir / "bin" / "mf6_latest_2026-04.exe" # Path to MODFLOW executable

shp_path = home_dir / "shapefiles" / "counties_5070.shp" # Path to GIS shapefile of Illinois counties
                                                            # Only used for visual reference on plots.

print(sim_name, base_model_workspace, exe_name, shp_path, sep="\n")
print(type(base_model_workspace))

## III. Loading the Base Model (Steady State)

MODFLOW 6 organizes its work into a **simulation** that can contain one or more models.
Our base model is a **steady-state** simulation — it represents long-term average groundwater conditions
with no time-varying stresses. This makes it an ideal starting point for understanding the aquifer system
before we introduce any perturbations (like drought).

### MODFLOW 6 Model Types
| Abbreviation | Full Name | Description |
|---|---|---|
| `gwf` | Groundwater Flow | Simulates groundwater heads and flows |
| `gwt` | Groundwater Transport | Simulates contaminant/solute transport |
| `prt` | Particle Tracking | Traces groundwater pathlines |
| `gwe` | Groundwater Energy | Simulate heat transport in groundwater |

For this analysis, we only use the **gwf** model.

In [ ]:
# MODFLOW 6 is structured as a "simulation" [sim] that can be composed of more than 1
# "model" (i.e. groundwater flow [gwf], groundwater transport [gwt],
# particle tracking [prt] etc). First let's load the full simulation of our base model.

base_sim = flopy.mf6.MFSimulation.load(
    sim_name=sim_name,
    sim_ws=base_model_workspace,
    write_headers=False,
    exe_name=exe_name
)

print("\n-- Success! --\n")

In [ ]:
# Now let's extract the groundwater flow model that we're interested in. 
# Spoilers, this simulation only has one model. 

base_model_name = list(base_sim.model_names)[0]

print(base_model_name)

base_gwf = base_sim.get_model(base_model_name)

print(Path(base_gwf.model_ws) == base_model_workspace)

## IV. Inspecting and Visualizing the Base Model

Now that the model is loaded, we can begin to explore its structure and properties.
We will look at:

1. **Grid dimensions** — how many rows, columns, and layers the model has
2. **Hydraulic conductivity (Kh)** — how easily water flows horizontally through each cell
3. **Layer geometry** — the top elevation and bottom elevations that define layer thickness
4. **Simulated heads** — the steady-state water levels produced by the base model
5. **Transmissivity (T)** — a combined measure of how much water an aquifer can transmit (T = Kh × saturated thickness)

### Grid Dimensions

The model grid divides the study area into rows and columns (horizontal) and layers (vertical).
Each cell represents a block of aquifer material. Knowing the grid dimensions is foundational
for correctly indexing and interpreting model arrays.

In [ ]:
# Determine model dimensions
nrow = base_gwf.modelgrid.nrow
ncol = base_gwf.modelgrid.ncol
nlay = base_gwf.modelgrid.nlay

print(f"# of model rows: {nrow}\n# of model columns: {ncol}\n# of model layers: {nlay}")

### Hydraulic Conductivity (Kh)

**Hydraulic conductivity** (Kh) describes how easily groundwater moves horizontally through a material.
It is stored in the Node Property Flow (NPF) package of the MODFLOW simulation.

Units are meters per day (m/d). Higher values indicate more permeable materials (e.g., coarse sand and gravel);
lower values indicate less permeable materials (e.g., clay or silt).

The `base_gwf.npf.k.array` call returns a 3D NumPy array with shape `(nlay, nrow, ncol)`.

In [ ]:
# Extract horizontal conductivity (Kh)
base_kh = base_gwf.npf.k.array

print(base_kh.shape)
print(base_kh[8, 100:201, 50:61])

### Plan-View Map of Hydraulic Conductivity

Below we plot Kh for **Layer 9 (index 8)** — the deepest unconsolidated aquifer layer.
This is the primary water supply layer for many municipalities in the Illinois study area.

Elements of this map:
- **Color flood**: spatial variation in Kh across the model domain
- **Red squares**: pumping wells screened in Layer 9
- **Cyan squares**: *G*eneral *H*ead *B*oundaries, representing the participation of Lake Michigan in our model
- **Orange outlines**: Illinois county boundaries (from a GIS shapefile)
- **Inactive cells** (cells outside the model domain) are masked out (e.g. far field stresses from deeper into the Lake Michigan interior that don't meaningfully contribute to our model solution or problem domain)

In [ ]:
# Create a "simple" map showing how horizontal conductivity varies spatially
# within the basal unconsolidated layer. This is where many public supply wells
# get their water from
with flopy.plot.styles.USGSMap():

    # Housekeeping
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_aspect('equal')

    # Create empty Map View from base model
    mapview = flopy.plot.PlotMapView(model=base_gwf, layer=8, ax=ax)
        
    # Mask inactive cells
    idomain = base_gwf.modelgrid.idomain
        
    # Apply the mask
    masked_kh_array = np.ma.masked_where(idomain == 0, base_kh)

    # Plot masked kh over model domain
    kh_color_flood = mapview.plot_array(masked_kh_array, ax=ax, cmap="managua", alpha=1.0, vmin=masked_kh_array.min(), vmax=masked_kh_array.max())
    cb = fig.colorbar(kh_color_flood, ax=ax, shrink=0.6)
    cb.set_label("Hydraulic Conductivity - Kh (m/d)")

    # Example to plot boundary conditions.
    # - Wells... they will be hard to see at the scale of the map.
    mapview.plot_bc("WEL", color="red")

    # - "Lake Michigan"... to simplify the model solution and speed up runtimes
    #    We "generalize" the big lake in the NE quadrant of our model domain as
    #    a region of cyan General Head Boundary "boundary condition" cells in layer 1
    mapview_ghb = flopy.plot.PlotMapView(model=base_gwf, layer=0, ax=ax)
    mapview_ghb.plot_bc("GHB", color="cyan")
    
    
    # Trick for custom legends
    ax.scatter([], [], c="red", marker="s", s=50, label="Pumping Wells (Layer 9)") # Create empty scatter plot, and only
                                                                                    # define the styling.
    ax.scatter([], [], c="cyan", marker="s", s=50, label="Lake Michigan (GHB, Layer 1)")
    
    ax.legend(loc="upper right")

    # Plot Illinois counties from shapefile for visual reference
    mapview.plot_shapefile(shp_path, ax=ax, facecolor="none", edgecolor="#FF5F05", linewidth=2.5)

    # Usually helpful for a final spit and polish.
    # Doesn't do much in this case. Probably because
    # the flopy.plot.styles.USGSMap() context manager
    # handles spit and polish in the background for us by default.
    # 
    # We'll keep it in, because it's good practice in most contexts.
    fig.tight_layout()

    # Don't forget to render the plot, after you make it.
    plt.show()
    

### Layer Geometry and Thickness

Each model layer has a defined **top** and **bottom** elevation. From these, we can compute the
total **layer thickness** for each cell:

$$\text{thickness}_{layer} = \text{top}_{layer} - \text{bottom}_{layer}$$

- For **Layer 1**, the top is the land surface elevation (`modelgrid.top`).
- For **deeper layers**, the top is the bottom of the layer above (`modelgrid.botm[layer-1]`).

This gives us a 3D array of cell thicknesses with shape `(nlay, nrow, ncol)`.

In [ ]:
model_top_elevation = base_gwf.modelgrid.top
layer_botm_elevations = base_gwf.modelgrid.botm

base_thickness = np.zeros_like(base_kh)

for layer in range(nlay):
    if layer == 0:
        base_thickness[layer] = model_top_elevation - layer_botm_elevations[layer]
    else:
        base_thickness[layer] = layer_botm_elevations[layer-1] - layer_botm_elevations[layer]

print(base_thickness[8])

### Simple Transmissivity

**Transmissivity (T)** is the rate at which groundwater can flow horizontally through a unit width of aquifer.
A simple estimate uses the full layer thickness, regardless of the water table position:

$$T_{simple} = K_h \times \text{total thickness}$$

This is valid for **confined** aquifer conditions (where the aquifer is fully saturated).
For **unconfined** (water-table) conditions, we need to use the *saturated* thickness instead — computed next.

In [ ]:
simple_transmissivity = base_kh * base_thickness

print(simple_transmissivity[8])

### Reading the Simulated Head Output

After MODFLOW runs, it writes the simulated groundwater heads to a binary `.hds` file.
FloPy's `output.head()` method reads this file into Python.

- `get_times()` returns a list of all output times in the simulation.
- `get_data(totim=...)` retrieves heads at a specific time.

We also replace any cells where the head exceeds `1e20` (MODFLOW's sentinel for dry or inactive cells)
with the layer bottom elevation, which is a defensible substitute for further calculations.

In [ ]:
base_head_file = base_gwf.output.head()
base_times = base_head_file.get_times()
base_heads = base_head_file.get_data(totim=base_times[-1])


base_heads = np.where(base_heads > 1e20, layer_botm_elevations, base_heads)

print(base_heads[8])

### Improved (Head-Based) Transmissivity

For **unconfined** (water-table) aquifer cells, the saturated thickness depends on where the water table
sits relative to the layer bottom — not the full layer thickness.

MODFLOW stores the cell type in the NPF package:
- `icelltype = 0` → **confined**: saturated thickness = full layer thickness
- `icelltype > 0` → **unconfined**: saturated thickness = head − layer bottom

We use `np.clip` to handle edge cases:
- If a cell is **dry** (head < bottom), clip to 0 (zero saturated thickness)
- If a head is **above** the layer top (artesian), clip to the full layer thickness

This head-based transmissivity is a better reflection of aquifer conditions under simulated steady-state.

In [ ]:
# Initiate a "saturated thickness" array as a copy of the base model thickness
# determined via layer elevations
saturated_thickness = np.copy(base_thickness)

# Get the cell type (0=confined, >0=unconfined)
base_icelltype = base_gwf.npf.icelltype.array

# Capture the unconfined model cells
unconfined_mask = base_icelltype > 0

# Determine saturated thickness globally based on model heads and layer elevations
head_thickness = base_heads - layer_botm_elevations

# Fix any dry cells where simulated model head fell below layer bottom
# as well as other model oddities
# Clip limits the values: minimum of 0 (dry), maximum of total thickness (fully saturated)
head_thickness = np.clip(head_thickness, 0, base_thickness)

# Apply head-based thickness ONLY where icelltype > 0
saturated_thickness[unconfined_mask] = head_thickness[unconfined_mask]

# Calculate a more refined transmissivity
better_transmissivity = base_kh * saturated_thickness

# print(simple_transmissivity - better_transmissivity)
# print((simple_transmissivity - better_transmissivity).max())
# print(np.unravel_index(np.argmax(simple_transmissivity - better_transmissivity), (simple_transmissivity - better_transmissivity).shape))
# print(np.histogram(simple_transmissivity - better_transmissivity))

In [ ]:
# Plotting properties
vmin = better_transmissivity.min() # minimum non-zero transmissivity in model
vmax = better_transmissivity.max() # maximum transmissivity in model

## IV. Plan-View Map with Cross-Section Lines

To explore the three-dimensional structure of the aquifer, we will create a series of vertical
**cross-sections** — slices through the model domain. Before plotting the cross-sections themselves,
we first define where those slices will be located and display them on a plan-view map.

Two types of cross-sections will be shown:
- **North–South (NS)**: slices running along model columns
- **East–West (EW)**: slices running along model rows

An `EDGE_BUFFER` keeps the cross-section lines away from the model boundary, where edge effects
can make results less representative.

In [ ]:
# Define a buffer around the model edges
EDGE_BUFFER = 5

# Specify number of cross sections in the North/South direction
NS_XSECT = 6

# Specify number of cross sections in the East/West direction
EW_XSECT = 7

In [ ]:
# N-S cross sections run along columns
col_indices = np.linspace(EDGE_BUFFER, ncol - 1 - EDGE_BUFFER, NS_XSECT, dtype=int)

# E-W cross sections run along rows
row_indices = np.linspace(EDGE_BUFFER, nrow - 1 - EDGE_BUFFER, EW_XSECT, dtype=int)

# -1 is needed because 0-based indices mean the last column[or row] is ncol [or nrow] - 1

#help(np.linspace)
print(f"N/S cross section columns: {col_indices}", f"E/W cross section rows: {row_indices}", sep="\n")

### Building the Cross-Section Definitions List

We store each cross-section as a dictionary in a list called `XSEC_DEFS`.
This makes it easy to loop over all cross-sections and plot them systematically.

Each dictionary has:
- `name`: a human-readable name used in plot titles
- `line`: a FloPy-style line definition specifying whether to slice by `Row` or `Column`
- `label`: a short label for axis annotations

In [ ]:
XSEC_DEFS = []

# N-S cross sections run along columns
for i, col in enumerate(col_indices):
    XSEC_DEFS.append({
        "name": f"North-South {i+1}",
        "line": {'Column': col},
        "label": f"Column {col}"
    })

# E-W cross sections run along rows
for i, row in enumerate(row_indices):
    XSEC_DEFS.append({
        "name": f"East-West {i+1}",
        "line": {'Row': row},
        "label": f"Row {row}"
    })

print(XSEC_DEFS)

## V. `plot_xs_plan_view_map` — Reusable Plotting Function

Before we commit to the computationally expensive task of computing `NS_XSECT + EW_XSECT` cross sections, let's contemplate a core programming and modeling principle/best practice: 

---

### 🌌 *Advice from the Ghost of Old, Ben Kenobi:* Look before you leap.

>We can use elements from the Star Wars prequel trilogy as allegory for the consequences of failing to look before we leap:

>  - **The overarching thematic mistake of the prequel era is the Jedi Council’s failure to foresee the growing darkness due to their own arrogance and political entrenchment.** Without knowing the true origin of the clone army (ordered by a deceased Jedi and provided by an enemy), the Jedi accept and use the army immediately, creating a situation where they are already compromised before the war begins. Furthermore, despite seeing clear red flags of anger and attachment in Anakin, the Council leaps into training him because of the prophecy, failing to "look" at the potential danger he poses.
 
>  - **Anakin’s character arc is defined by jumping into action without considering the long-term consequences, driven by emotion rather than wisdom.** Throughout *Attack of the Clones* and *Revenge of the Sith*, Anakin consistently ignores the advice of Obi-Wan Kenobi, jumping into dangerous situations (e.g., chasing Zam Wesell, confronting Dooku alone).

>  - **The Republic "leaps" into granting emergency powers to Chancellor Palpatine without looking at the long-term danger of authoritarianism.** To handle the Clone Wars, the Senate immediately embraces militarization and centralizes power in Palpatine, effectively voting away their own democracy.  The Clone Wars, themselves, are engineered to act as a distraction, keeping the Jedi fighting on the front lines so they cannot "look" at the political corruption happening in their own capital.

---


### HowTo on *FUN*ctions in Python... 
Instead of finding out later that the cross sections we chose aren't very interesting, or that we haven't satisfactorily covered our model domain, or we missed an especially interesting part of our model domain that we want to investigate further... Let's make a customized plan view map of our model domain, that shows the traces of our proposed cross sections alongside model and geospatial visual references.

Since we don't want to do a lot of copy and paste or tedious code modifications to switch between plotting Kh [`base_kh`] or T [`better_transmissivity`], let's write a template for a "subroutine" that will take in custom parameters and generate different maps based on our preferences. In Python, one way to write subroutine templates is via the concept of functions. An example of the syntax involved in declaring functions is:

```python
def apply_for_research_grant(walk_name, steps_taken): # def function_name(parameter1, parameter2 ... etc):
    """
    Evaluates if a walk is silly enough for government funding.
    """
    # Define a 'Silly' Database (A List)
    # Lists store multiple items in one variable.
    approved_styles = ["The Double-Up", "The Sideways Hop", "The Tea-Tray Glide"]

    # Bureaucratic Assessment (Logic)
    if walk_name in approved_styles and steps_taken > 10:
        decision = f"The Ministry approves! Your {walk_name} is spectacularly absurd."
    
    elif walk_name not in approved_styles:
        # If the walk is too normal (like a common stroll)
        decision = "It's not particularly silly, is it? Just a walk, really."
    
    else:
        decision = "The spirit is there, but we need more steps to verify the silliness."

    # Return the decision for further processing
    # Note... not every function has to return something.
    # Sometimes we write functions to automate workflows, 
    # and standardize procedures.
    return decision

# Testing the grant application:
my_application = apply_for_research_grant("The Double-Up", 15)
print(f"Official Notice: {my_application}")
```

The function below creates a plan-view map that:
1. Displays a 3D model array (here: transmissivity) as a color flood on a selected layer
2. Overlays the cross-section lines with labels
3. Optionally overlays a background shapefile (county boundaries)
4. Optionally saves the figure to disk

By packaging this logic into a function, we can reuse it easily for different arrays or models
without copy-pasting code. Run the `help()` cell below to see the full parameter documentation.

Once we're happy with the locations and density of cross sections that we want to generate. We will use another function to plot them :).

In [ ]:
def plot_xs_plan_view_map(model,
                              array_to_plot,
                              row_indices,
                              col_indices,
                              edge_buffer,
                              shapefile_path=None,
                              layer=8,
                              vmin=0,
                              vmax=100,
                              cbar_label="Units not specified",
                              save_path=None
                         ):
    """
    Plots a plan view map of a Flopy groundwater model, including a user-provided 
    3D array (excluding inactive cells), an optional background shapefile 
    (Illinois counties) and labeled cross-section lines.

    Parameters
    ----------
    model : flopy.mf6.ModflowGwf
        The loaded Flopy groundwater flow model object.
    array_to_plot : numpy.ndarray
        The 3D array of data to visualize (e.g., Transmissivity, Heads).
    row_indices : list or numpy.ndarray
        Array of row indices where the East-West cross-sections are located.
    col_indices : list or numpy.ndarray
        Array of column indices where the North-South cross-sections are located.
    edge_buffer : int
        The number of cells to buffer away from the model edges when drawing the lines.
    shapefile_path : pathlib.Path or str, optional
        Path to the shapefile to plot over the model domain. Default is None.
    layer : int, optional
        The zero-based model layer to visualize. Default is 8 [the lowermost unconsolidated layer].
    vmin : float or int, optional
        The minimum value for background color flood. Default is 0.
    vmax : float or int, optional
        The maximum value for background color flood. Default is 100.
    cbar_label : str, optional
        The text label for the colorbar. Default is "Units not specified".
    save_path : pathlib.Path or str, optional
        Path to save the output figure. If None, the figure is not saved. Default is None.

    Returns
    -------
    None
        Renders and displays the Matplotlib map plot.
    """
    
    with flopy.plot.styles.USGSMap():
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.set_aspect('equal')
        
        mapview = flopy.plot.PlotMapView(model=model, layer=layer, ax=ax)
        
        # Masking inactive cells
        idomain = model.modelgrid.idomain
        
        # Apply the mask directly to the user-provided array
        masked_array = np.ma.masked_where(idomain == 0, array_to_plot)
        
        csa = mapview.plot_array(masked_array, ax=ax, cmap="managua", vmin=vmin, vmax=vmax)
        cb = fig.colorbar(csa, ax=ax, shrink=0.6)
        
        # Use the dynamic colorbar label
        cb.set_label(cbar_label)

        # Example to plot boundary conditions.
        # Wells in this case... they will be hard to see at the scale of the map.
        mapview.plot_bc("WEL", color="red")

        # - "Lake Michigan"... to simplify the model solution and speed up runtimes
        #    We "generalize" the big lake in the NE quadrant of our model domain as
        #    a region of cyan General Head Boundary "boundary condition" cells in layer 1
        mapview_ghb = flopy.plot.PlotMapView(model=base_gwf, layer=0, ax=ax)
        mapview_ghb.plot_bc("GHB", color="cyan")

        # Trick for custom legends... needs work in this context
        ax.scatter([], [], c="red", marker="s", s=50, label="Pumping Wells (Layer 9)") # Create empty scatter plot, and only                                                                            # define the styling.
        ax.scatter([], [], c="cyan", marker="s", s=50, label="Lake Michigan (GHB, Layer 1)")
        ax.legend(loc="upper right")
        
        if shapefile_path is not None:
            mapview.plot_shapefile(
                str(shapefile_path),
                ax=ax,
                facecolor="none",
                edgecolor="#FF5F05",
                linewidth=2.5,
            )
            
        mg = model.modelgrid
        nrow, ncol = mg.nrow, mg.ncol
        bbox_style = dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="black")

        # -------------------------------------------------------------
        # Plot East-West (Row) Cross-Section Lines
        # -------------------------------------------------------------
        for i, row in enumerate(row_indices):
            label_num = i + 1
            
            # -1 is needed because 0-based indices mean the last column is ncol - 1
            start_col = edge_buffer 
            end_col = ncol - 1 - edge_buffer 
            
            x0 = mg.xcellcenters[row, start_col]
            y0 = mg.ycellcenters[row, start_col]
            x1 = mg.xcellcenters[row, end_col]
            y1 = mg.ycellcenters[row, end_col]
            
            ax.plot([x0, x1], [y0, y1], color="black", linestyle="--", linewidth=1.5)
            
            ax.annotate(f"EW{label_num}", (x0, y0), textcoords="offset points", xytext=(-8, 0), 
                        ha='right', va='center', fontweight='bold', bbox=bbox_style)
            ax.annotate(f"EW{label_num}'", (x1, y1), textcoords="offset points", xytext=(8, 0), 
                        ha='left', va='center', fontweight='bold', bbox=bbox_style)

        # -------------------------------------------------------------
        # Plot North-South (Column) Cross-Section Lines
        # -------------------------------------------------------------
        for i, col in enumerate(col_indices):
            label_num = i + 1
            
            # Use the edge_buffer variable dynamically
            start_row = edge_buffer
            end_row = nrow - 1 - edge_buffer 
            
            x0 = mg.xcellcenters[start_row, col]
            y0 = mg.ycellcenters[start_row, col]
            x1 = mg.xcellcenters[end_row, col]
            y1 = mg.ycellcenters[end_row, col]
            
            ax.plot([x0, x1], [y0, y1], color="black", linestyle="--", linewidth=1.5)
            
            ax.annotate(f"NS{label_num}", (x0, y0), textcoords="offset points", xytext=(0, 8), 
                        ha='center', va='bottom', fontweight='bold', bbox=bbox_style)
            ax.annotate(f"NS{label_num}'", (x1, y1), textcoords="offset points", xytext=(0, -8), 
                        ha='center', va='top', fontweight='bold', bbox=bbox_style)

        # -------------------------------------------------------------
        # Fancy Pants Post Processing to Make the Plot Look Pretty
        # -------------------------------------------------------------
        
        # Get the current tight axis limits set by Flopy
        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()
        
        # Calculate a 10% spatial buffer based on the domain extent
        x_padding = (xmax - xmin) * 0.10
        y_padding = (ymax - ymin) * 0.10
        
        # Apply the expanded limits to create whitespace around the model
        ax.set_xlim(xmin - x_padding, xmax + x_padding)
        ax.set_ylim(ymin - y_padding, ymax + y_padding)

        # Add 'pad' and 'labelpad' to push text away from the tick marks
        ax.set_title(f"Model Domain Plan View (Layer {layer+1})", pad=20)
        ax.set_xlabel("Easting", labelpad=10)
        ax.set_ylabel("Northing", labelpad=10)
        
        # Tell Matplotlib to dynamically rearrange elements to fit the figure box
        fig.tight_layout()

        # Standard method for exporting figures...
        # Recast user provided save_path as a Path object
        # Check that the parent directory exists. If it doesn't...
        # Use Path methods [mkdir] to *m* a *k* e the *dir* ectory
        # exist.
        if save_path is not None:
            out_path = Path(save_path)
            out_path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(out_path, dpi=300, bbox_inches="tight")

        # Render the plot to the screen
        plt.show()

In [ ]:
help(plot_xs_plan_view_map)

The cell below calls `plot_xs_plan_view_map` using the **head-based transmissivity** array we calculated earlier.
The dashed lines show where the N-S and E-W cross-sections will be taken.

In [ ]:
plot_xs_plan_view_map(model=base_gwf,
                        array_to_plot=better_transmissivity,
                        row_indices=row_indices,
                        col_indices=col_indices,
                        edge_buffer=EDGE_BUFFER,
                        shapefile_path=shp_path,
                        # layer=8, #optional parameter, we're happy with the default
                        vmin=vmin,
                        vmax=vmax,
                        cbar_label="Transmissivity (m\u00b2/day)",
                        save_path=None
                    )

## VI. Vertical Cross-Sections

Vertical cross-sections cut through the model in the N-S or E-W direction and show how
aquifer properties vary with depth across the study area.

The `plot_custom_cross_section` function below:
1. Uses FloPy's `PlotCrossSection` to project the 3D model array onto a 2D vertical slice
2. Overlays the **ibound** boundary (active vs. inactive cells)
3. Overlays **pumping well** locations where they intersect the cross-section

Each cross-section is defined by a dictionary (`xs_def`) that specifies whether to slice
along a **Row** (E-W) or **Column** (N-S), along with display labels.

In [ ]:
def plot_custom_cross_section(xs_def,
                              model,
                              array_to_plot,
                              vmin,
                              vmax,
                              cbar_label="Units not specified",
                              save_path=None
                             ):
    """
    Plots a 2D cross-section of a 3D model array using Flopy and Matplotlib.

    Parameters
    ----------
    xs_def : dict
        A dictionary defining the cross-section. Must contain:
        - 'line': dict specifying the row or column (e.g., {'Row': 5}).
        - 'name': str containing the descriptive name of the cross-section.
        - 'label': str containing the short label (e.g., 'Row 5').
    model : flopy.mf6.ModflowGwf
        The loaded Flopy groundwater flow model object.
    array_to_plot : numpy.ndarray
        The 3D array of data to visualize (e.g., hydraulic conductivity, transmissivity).
    vmin : float or int
        The minimum value for the color mapping.
    vmax : float or int
        The maximum value for the color mapping.
    cbar_label : str, optional
        The text label for the colorbar. Default is "Units not specified".
    save_path : pathlib.Path or str, optional
        Path to save the output figure. If None, the figure is not saved. Default is None.

    Returns
    -------
    None
        Renders and displays the Matplotlib plot.
    """
    
    # Use the Flopy style context manager for a professional USGS-style look
    with flopy.plot.styles.USGSPlot():
        fig = plt.figure(dpi=300, figsize=(8, 4)) # Slightly larger for the styled plot
        ax = fig.add_subplot(1, 1, 1)

        # Initialize PlotCrossSection using the dynamic line definition
        xsect = flopy.plot.PlotCrossSection(
            model=model,
            ax=ax,
            line=xs_def["line"] 
        )

        # Plot the requested array
        csa = xsect.plot_array(
            array_to_plot,
            cmap="managua",
            vmin=vmin,
            vmax=vmax,
        )

        xsect.plot_ibound() # Shows inactive/active boundaries clearly
        xsect.plot_bc("WEL") # Shows where the wells are located
        xsect.plot_bc("SFR") # Shows where higher order streams and rivers are located
        xsect.plot_bc("GHB") # Shows where Lake Michigan General Head Boundaries are located

        # Add the colorbar to the figure, linked to the axis
        cb = fig.colorbar(csa, ax=ax, shrink=0.8)
        cb.set_label(cbar_label)

        # Set dynamic title and labels
        ax.set_title(f"{xs_def['name']} ({xs_def['label']})")
        ax.set_xlabel("Distance (m)")
        ax.set_ylabel("Elevation (m)")

        # Handle optional saving
        if save_path is not None:
            # Convert to a Path object to ensure cross-platform compatibility
            out_path = Path(save_path)
            # Create the parent directory if it doesn't exist yet
            out_path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(out_path, dpi=800, bbox_inches="tight")
        
        plt.show()

In [ ]:
def plot_custom_cross_section(xs_def,
                                  model,
                                  array_to_plot,
                                  vmin,
                                  vmax,
                                  cbar_label="Units not specified",
                                  save_path=None
                             ):
    """
    Plots a 2D cross-section of a 3D model array using Flopy and Matplotlib.

    Parameters
    ----------
    xs_def : dict
        A dictionary defining the cross-section. Must contain:
        - 'line': dict specifying the row or column (e.g., {'Row': 5}).
        - 'name': str containing the descriptive name of the cross-section.
        - 'label': str containing the short label (e.g., 'Row 5').
    model : flopy.mf6.ModflowGwf
        The loaded Flopy groundwater flow model object.
    array_to_plot : numpy.ndarray
        The 3D array of data to visualize (e.g., hydraulic conductivity, transmissivity).
    vmin : float or int
        The minimum value for the color mapping.
    vmax : float or int
        The maximum value for the color mapping.
    cbar_label : str, optional
        The text label for the colorbar. Default is "Units not specified".
    save_path : pathlib.Path or str, optional
        Path to save the output figure. If None, the figure is not saved. Default is None.

    Returns
    -------
    None
        Renders and displays the Matplotlib plot.
    """
    
    # Use the Flopy style context manager for a professional USGS-style look
    with flopy.plot.styles.USGSPlot():
        fig = plt.figure(dpi=300, figsize=(8, 4)) # Slightly larger for the styled plot
        ax = fig.add_subplot(1, 1, 1)

        # Initialize PlotCrossSection using the dynamic line definition
        xsect = flopy.plot.PlotCrossSection(
            model=model,
            ax=ax,
            line=xs_def["line"] 
        )

        # Plot the requested array
        csa = xsect.plot_array(
            array_to_plot,
            cmap="managua",
            vmin=vmin,
            vmax=vmax,
        )

        xsect.plot_ibound() # Shows inactive/active boundaries clearly
        xsect.plot_bc("WEL") # Shows where the wells are located
        xsect.plot_bc("SFR") # Shows where the high order streams and rivers are located
        xsect.plot_bc("GHB") # Shows where the Lake Michigan General Head Boundaries are located

        # Add the colorbar to the figure, linked to the axis
        cb = fig.colorbar(csa, ax=ax, shrink=0.8)
        cb.set_label(cbar_label)

        # Set dynamic title and labels
        ax.set_title(f"{xs_def['name']} ({xs_def['label']})")
        ax.set_xlabel("Distance (m)")
        ax.set_ylabel("Elevation (m)")

        # Handle optional saving
        if save_path is not None:
            # Convert to a Path object to ensure cross-platform compatibility
            out_path = Path(save_path)
            # Create the parent directory if it doesn't exist yet
            out_path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(out_path, dpi=800, bbox_inches="tight")
        
        plt.show()

### Plotting All Cross-Sections

The loop below iterates through every cross-section definition and calls `plot_custom_cross_section`.
This produces a series of N-S and E-W vertical slices of the **transmissivity** field,
giving us a three-dimensional picture of how the aquifer's productivity varies across Illinois.

> **Tip**: To save each cross-section to a file, update `save_dir` (save directory) to a valid Path( ) object.
> 
> Like `home_dir` !
>
> or `base_model_workspace / "postproc"` !

In [ ]:
save_dir = None

# save_dir = base_model_workspace / "postproc"

In [ ]:
for xs_def in XSEC_DEFS:
    
    if save_dir is not None:
        clean_name = xs_def['name'].replace(' ', '_')
        save_path =  save_dir / f"xsec_{clean_name}.png"
    else: save_path = None
    
    plot_custom_cross_section(
        xs_def=xs_def,
        model=base_gwf,
        array_to_plot=better_transmissivity,
        vmin=vmin,
        vmax=vmax,
        cbar_label="Transmissivity (m\u00b2/day)",
        save_path=save_path
    )

## *"If you strike me down, I shall become more powerful than you can possibly imagine."*

In this notebook, you:

1. ✅ Loaded the IISG MODFLOW 6 base model using FloPy
2. ✅ Inspected grid dimensions, hydraulic conductivity, and layer geometry
3. ✅ Computed both simple and head-based **Transmissivity**
4. ✅ Visualized model properties in plan-view maps and vertical cross-sections

Whichever path we decide to take next, our basic workflow will remain the same.
 - Read in the base model
 - Strip it for parts
 - Enhance existing components
 - Add new features, relevant to the question at hand
 - Run an updated model
 - Interpret results

We're all in for a hard lesson about letting go of attachments and challenging our intuition... Especially Daniel who worked so hard to craft our base model in the first place... but the models we get to build are truly *more powerful than you can possibly imagine.*

**Next steps**:

 - Proceed to `1_new_demands.ipynb` to modify this base model and simulate the effects of drilling new wells to accommodate local increased water demands.
   - This exercise is the most "participatory" of the three. You, the audience, will be confronted with decisions to make, and limited (but helpful) information with which to make those decisions. We, your friendly neighborhood scientists, will do our best to follow your guidance, and then we will explore the results together. "Choose your own adventure", with millions of possible outcomes.

>Vote **(1)**, especially if your favorite Star Wars Original Trilogy film is *A New Hope* (1977)
***

 - Proceed to `2_drought_impacts.ipynb` to modify this base model and simulate the effects of a year of **intense** drought on regional groundwater levels.
   - Who wants to be burdened with high-stakes decision-making first thing on a Monday morning? Unlike `new_demands.ipynb` , *`drought_impacts.ipynb`* and `gw_age.ipynb` are more of a "sit back, relax, and enjoy the show" proposition. There will be plenty of opportunities to ask questions along the way, and perhaps an opportunity or two to "alter the final outcome", but less stress and active effort is required from participants.

>Vote **(2)**, especially if your favorite Star Wars Original Trilogy film is *The Empire Strikes Back* (1980)
***
 
 - Proceed to `3_gw_age.ipynb` to modify this base model and simulate groundwater age relationships and multi-year contamination capture zones for high-capacity water wells.
   - Get ready to have your mind blown to smithereens... much like Death Star II at the end of the final film of the original Star Wars trilogy. Sip on some coffee, and be amazed at what these modeling tools can do in the hands of a true "Jedi Master", while still being able to grasp the basic workflow and opportunities to use and modify these tools yourself.

>Vote **(3)**, especially if your favorite Star Wars Original Trilogy film is *Return of the Jedi* (1983)